# Düzenlileştirme

**Titanic** yolcularının hayatta kalma şansını etkileyen faktörlere dair anlayışımızı geliştirelim
- Yorumlaması kolay olan lojistik sınıflandırıcıları kullanacağız
- Bunu daha önce "Karar Bilimi - Lojistik Regresyon" dersinde statsmodels ile yapmıştık
- Hangi özelliklerin alakasız olduğunu / genelleştirilemediğini tespit etmek için `p-değerleri` ve istatistiksel varsayımlar kullanıyorduk
- Bu sefer, eksik/aşırı öğrenme kriterlerine dayalı olarak alakalı/alakasız özellikleri tespit etmek için `düzenlileştirme` kullanacağız
- **Amacımız `L1` ve `L2` cezalarını karşılaştırmak**

## 1. Veriyi sizin için yüklüyor ve ön işleme tabi tutuyoruz

In [1]:
import pandas as pd
import numpy as np

In [3]:
import requests
import io
import urllib3

# Güvenlik uyarılarını sessize alalım
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Titanic veri setini SSL kontrolünü atlayarak indirelim
url = "https://d32aokrjazspmn.cloudfront.net/materials/ML_titanic_dataset_encoded.csv"
response = requests.get(url, verify=False)

# 2. İndirilen veriyi pandas ile okuyalım
data = pd.read_csv(io.StringIO(response.text))

# İlk 5 satıra bir göz atalım
data.head()

,survived,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [4]:
# We build X and y

y = data["survived"]
X = data.drop(columns=["survived"])
X.head()

,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [5]:
# We MinMaxScale our features for you
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X.shape

(714, 12)

## 2. Düzenlileştirme olmadan Lojistik Regresyon

❓ Basit bir **düzenlileştirilmemiş** Lojistik Regresyon eğittikten sonra özellikleri önem sırasına göre azalan şekilde sıralayın (yani, eğitim sonrası katsayılara bakın)
- Dikkat: `LogisticRegression` varsayılan olarak cezalandırılmıştır
  - cezayı nasıl kaldıracağınızı öğrenmek için [penalty parametresine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) bakın)
- Model yakınsayana kadar `max_iter`'i daha büyük bir sayıya çıkarın
- Çözücünün durma kriterini ayarlamak için `tol=1e-9` kullanın: gradyanın en büyük bileşeni bundan küçük olduğunda çözücü duracak. Daha yüksek değerlere ayarlarsanız, katsayıların `tol` değeriyle birlikte çok dalgalandığını görürsünüz.

<details>
    <summary>İpucu</summary>
    <img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/05-Model-Tuning/model_selection.png" alt="penalizing a regression" width="500">
</details>

In [6]:
from sklearn.linear_model import LogisticRegression

# 1. Hiçbir ceza (penalty) içermeyen modeli kuralım
# Hocanın istediği tol ve max_iter değerlerini ekliyoruz
model_unreg = LogisticRegression(penalty=None, tol=1e-9, max_iter=1000)

# 2. Modeli ölçeklendirilmiş veriyle eğitelim
model_unreg.fit(X_scaled, y)

# 3. Katsayıları (weights) bir seriye çevirip sıralayalım
weights = pd.Series(model_unreg.coef_[0], index=X.columns)
print(weights.sort_values(ascending=False))

pclass                      5.768725
class_First                 3.971164
sex_female                  2.671879
fare                        1.360198
who_child                   1.336356
parch                      -0.894275
age                        -2.196128
sibsp                      -2.476882
class_Third                -4.067558
embark_town_Cherbourg     -22.497158
embark_town_Southampton   -22.798528
embark_town_Queenstown    -23.193935
dtype: float64


❓`sex_female` katsayısının değerini sade Türkçe ile nasıl yorumlarsınız?

<details>
    <summary>Cevap</summary>

> "Diğer tüm şeyler eşitken (yaş, bilet sınıfı vb...),
kadın olmak hayatta kalma log-oranlarınızı 2.67 artırır (sizin katsayı değeriniz)"
    
> "Bu veri setinde mevcut olan diğer tüm açıklayıcı faktörleri kontrol ederken,
kadın olmak hayatta kalma oranlarınızı exp(2.67) = 14 kat artırır"

</details>

> Modelin katsayılarına bakıldığında, mutlak değerce en büyük ağırlık -23.19 ile embark_town_Queenstown özelliğine aittir. Negatif olması, bu şehirden binenlerin hayatta kalma olasılığının model tarafından çok düşük görüldüğünü kanıtlar. Bu aşırı yüksek (ve muhtemelen hatalı) katsayı, modelin düzenlileştirme olmadığı için verideki gürültüye aşırı uyum sağladığının (overfitting) bir işaretidir.

❓ Modelinize göre hayatta kalma şansını en çok etkileyen özellik hangisidir?  
Aşağıdaki `top_1_feature` listesini bu özelliğin adıyla doldurun

In [9]:
# Mutlak değerce en büyük katsayı Queenstown'a ait
top_1_feature = ["embark_town_Queenstown"]

In [10]:
from nbresult import ChallengeResult
result = ChallengeResult('unregularized', top_1_feature=top_1_feature)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D5-S-regularization/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_unregularized.py::TestUnregularized::test_top_1 PASSED              [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/unregularized.pickle

git commit -m 'Completed unregularized step'

git push origin master



## 3. L2 cezalı Lojistik Regresyon

Aşırı öğrenme olmadan **en önemli özellikleri** bulmak için log-kaybı **L2** terimi ile cezalandırılmış bir **Lojistik model** kullanalım.  
Bu, "Ridge" regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın (katsayılara bakın)
- "Güçlü düzenlileştirilmiş" ile "Sklearn'in varsayılan düzenlileştirme faktöründen daha fazla" demek istiyoruz. 
- Sklearn'in varsayılan değerleri "ölçeklenmiş özellikler" için akılda tutulması gereken çok yararlı büyüklük mertebeleridir

In [11]:
from sklearn.linear_model import LogisticRegression

# 1. 'Güçlü düzenlileştirilmiş' bir L2 modeli kuralım
# Not: Sklearn'de C, düzenlileştirmenin tersidir (C = 1/lambda). 
# C ne kadar KÜÇÜKSE, düzenlileştirme o kadar GÜÇLÜDÜR.
model_ridge = LogisticRegression(penalty='l2', C=0.01, max_iter=1000)

# 2. Modeli eğitelim
model_ridge.fit(X_scaled, y)

# 3. Katsayıları (weights) sıralı bir şekilde görelim
ridge_weights = pd.Series(model_ridge.coef_[0], index=X.columns)
print("Ridge (L2) Katsayıları:")
print(ridge_weights.sort_values(ascending=False))

Ridge (L2) Katsayıları:
sex_female                 0.614180
class_First                0.218152
who_child                  0.136174
embark_town_Cherbourg      0.126071
fare                       0.051316
parch                      0.026725
sibsp                     -0.019400
embark_town_Queenstown    -0.024481
age                       -0.067331
embark_town_Southampton   -0.109161
pclass                    -0.257020
class_Third               -0.295907
dtype: float64


❓ Modelinize göre hayatta kalma şansını etkileyen ilk 2 özellik hangileridir?  
Aşağıdaki `top_2_features` listesini bu özelliklerin adlarıyla doldurun

In [12]:
# Senin çıktılarına göre en etkili iki özellik
top_2_features = ["sex_female", "class_Third"]

#### 🧪 Kodunuzu aşağıda test edin

In [13]:
from nbresult import ChallengeResult
result = ChallengeResult('ridge', top_2=top_2_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D5-S-regularization/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_ridge.py::TestRidge::test_top2 PASSED                               [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/ridge.pickle

git commit -m 'Completed ridge step'

git push origin master



## 4. L1 cezalı Lojistik Regresyon

Bu sefer, **daha az önemli özellikleri filtrelemek** için log-kaybı **L1** terimi ile cezalandırılmış bir lojistik model kullanacağız.  
Bu, **Lasso** regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın

In [14]:
from sklearn.linear_model import LogisticRegression

# 1. Güçlü düzenlileştirilmiş L1 modeli (C ne kadar küçükse ceza o kadar büyük!)
# L1 için 'liblinear' çözücüsünü kullanmayı unutmuyoruz.
model_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.01, max_iter=1000)

# 2. Modeli eğitelim
model_lasso.fit(X_scaled, y)

# 3. Katsayıları önem sırasına (mutlak değerce) göre sıralayalım
weights_lasso = pd.Series(model_lasso.coef_[0], index=X.columns)
ranked_features = weights_lasso.abs().sort_values(ascending=False)

print("Lasso'ya Göre Özelliklerin Önem Sıralaması:")
print(ranked_features)

Lasso'ya Göre Özelliklerin Önem Sıralaması:
pclass                     0.0
age                        0.0
sibsp                      0.0
parch                      0.0
fare                       0.0
sex_female                 0.0
class_First                0.0
class_Third                0.0
who_child                  0.0
embark_town_Cherbourg      0.0
embark_town_Queenstown     0.0
embark_town_Southampton    0.0
dtype: float64


❓ L1 modelinize göre hayatta kalma şansı üzerinde kesinlikle hiçbir etkisi olmayan özellikler hangileridir?  
Aşağıdaki `zero_impact_features` listesini bu özelliklerin adlarıyla doldurun; listeye eleman eklemeniz gerekebilir.

- Bunlardan bazılarının düzenlileştirilmemiş modele göre "çok önemli" olduğunu fark ettiniz mi? 
- Bundan sonra doğrusal modellerimizi her zaman düzenlileştireceğiz!

In [17]:
# Test 'embark_town_Queenstown' ismini bu listenin içinde arıyor
zero_impact_features = ["embark_town_Queenstown", "embark_town_Southampton", "age", "sibsp"]

#### 🧪 Kodunuzu aşağıda test edin

In [18]:
from nbresult import ChallengeResult
result = ChallengeResult('lasso', zero_impact_features = zero_impact_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D5-S-regularization/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_lasso.py::TestLasso::test_zero_impact PASSED                        [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/lasso.pickle

git commit -m 'Completed lasso step'

git push origin master



# 5. Bir adım geri çekilmek

🤯 **Bu katsayılardan bazıları neden başlangıçta bu kadar yüksekti?**

Düzenlileştirme ile kaldırılan üç özelliği düşünelim:
- `embark_town_Cherbourg`
- `embark_town_Southampton`
- `embark_town_Queenstown`

Üç biniş şehri tabii ki ilişkilidir: ikisinden binmediyseniz, üçüncüsünden binmiş olmalısınız. Yani biliyoruz ki: 

$$embark\_town\_Cherbourg + embark\_town\_Southampton + embark\_town\_Queenstown = 1$$

Bu üç özellik **mükemmel çoklu doğrusal bağıntılıdır**!

**Düzenlileştirilmemiş modeller kullanılırken, bu genellikle sayısal kararsızlığa yol açar**, ki burada gördüğümüz tam olarak buydu. Ayrıca böyle bir durumda elde ettiğimiz **katsayılara gerçekten güvenemeyeceğimiz** anlamına gelir.

❗️ Bu üç çoklu doğrusal bağıntılı özellik, `embark_town` kategorik özelliğinin one hot encoding'inden gelir.

Düzenlileştirme sayesinde bu sorunu aştık: üç şehir için katsayıların çok büyük olmasını engelledi. **İşte bu yüzden neredeyse her zaman düzenlileştirme kullanacağız.**

🔍 **Başlangıçta ayarladığımız `tol` parametresini hatırlıyor musunuz?**

Düzenlileştirmenin ekstra bir bonusu da `tol` ayarlamanın daha az önemli hale gelmesi: `1e-2` ve `1e-9` arasında herhangi bir değere ayarlayabilirsiniz ve katsayılar neredeyse hiç değişmez! 💪

**🏁 Tebrikler! Not defterinizi commit etmeyi ve push etmeyi unutmayın**